# vault

> Memory that outlives the process: one vishalakshi vault behind `Host`, and the standing
> watches that put things back on the agent's desk.

In [ ]:
#| default_exp vault

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import time, tempfile
from pathlib import Path
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json, threading, time
from pathlib import Path
from fastcore.basics import AttrDict
from ramabana.core import AgentError, agent_err
from ramabana.tools import Hit, LocalHost, clip

## Why a vault

`LocalHost` can already read the web and index the repo, and neither survives the process. A
page read in one session is gone by the next, and the code index and the reading are two
unrelated stores that never see each other's results.

A [vishalakshi](https://vedicreader.github.io/vishalakshi/) `Vault` is one SQLite file holding
both. Putting it behind `Host` buys three things the harness cannot get any other way:

- **`memory_search` actually has something to search.** `LocalHost` raises
  `NotImplementedError` for the whole memory group, so `tools_for` drops it. With a vault the
  five memory tools appear, and they answer from everything ever read.
- **One ranking instead of three.** `search_code` fuses the vault's prose, kosha's identifiers
  and ripgrep's literals through `federate`. The legs share no vector space, so they are merged
  by rank rather than by distance.
- **Standing interests.** A vault has watches, so `watch_tools` appears too: the agent can
  arrange to re-read a page next week, or be reminded of something it worked out today.

In [ ]:
#| export
#: `Vault(None)` means `~/.vishalakshi/vault.db` -- the same vault the `vishalakshi` CLI uses,
#: which is the point: what you read at a shell prompt is what the agent can recall.
DFLT_VAULT = None
MEM_SECTIONS = 6          # operative sections a memory search returns
TOC_DEPTH = 2             # heading levels of a document's tree worth showing at once

In [ ]:
#| export
def _trim(node, depth=TOC_DEPTH):
    "One heading tree, cut to `depth` levels: a whole tree is a document, not an index entry."
    if not isinstance(node, dict): return node
    out = {k: node.get(k) for k in ('id', 'title', 'pages') if node.get(k) is not None}
    kids = node.get('children') or []
    if depth > 0 and kids: out['children'] = [_trim(k, depth - 1) for k in kids]
    elif kids: out['children'] = f'{len(kids)} more'
    return out


def _sect(s):
    "One retrieved section, cut to what a model needs: what it says, where it is, how to read it."
    return {k: s[k] for k in ('node_id', 'doc_id', 'title', 'breadcrumb', 'text') if k in s}


def _fed_hit(h):
    """One federated row as a `Hit`, keeping the handle that reopens it.

    The legs return different things -- a repo hit is a file and a line, a prose hit is a
    section of something read months ago -- so `symbol` carries whichever identifier the
    follow-up tool needs: a `mod_name` for `symbols`, a `node_id` for `memory_read`.
    """
    where = str(h.get('where') or h.get('ref') or '')
    path, _, num = where.rpartition(':')
    if h.get('source') == 'prose': return Hit(where, 1, str(h.get('ref') or ''), str(h.get('text') or '')[:200])
    return Hit(path if num.isdigit() else where, int(num) if num.isdigit() else 1,
               str(h.get('ref') or h.get('title') or ''), str(h.get('text') or '')[:200])

## The host

In [ ]:
#| export
class VaultHost(LocalHost):
    """`LocalHost` with a vishalakshi vault behind it: durable memory, federated search, watches.

    Everything `LocalHost` does still works the same way. What changes is that reading is no
    longer throwaway -- a page read through `read_url` is filed, `research` answers out of the
    file rather than out of a response body, and `search` asks the vault, the code index and
    ripgrep the same question at once.

    The vault is opened in a background thread, for the same reason the kosha sync is: it loads
    an embedding model, and the first thing that touches it is `tools_for` working out whether
    the memory tools should exist at all. Nobody should wait through a model load to find out
    that the answer is yes -- which is why the answer is `capabilities` and not a probe. A
    probe would have called `memory_tree('')`, which reaches `self.vault`, which blocks on the
    lock the warm thread is holding: the background open, waited through in full, by the one
    caller it was added for.
    """

    @property
    def capabilities(self):
        "Memory and watches, by construction: a `VaultHost` is the host that has a vault."
        return {**super().capabilities, 'memory': True, 'watch': True}

    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 vault=DFLT_VAULT,      # a `Vault`, a path to one, or None for ~/.vishalakshi/vault.db
                 federate=True,         # fuse vault prose into `search_code` alongside kosha and ripgrep
                 remember_reads=True,   # file what `read_url` fetches, so the next session has it
                 warm=True,             # open the vault in the background at construction
                 **kw):                 # forwarded to `LocalHost`
        super().__init__(roots, **kw)
        self._vault, self._vlock, self._vthread = vault, threading.Lock(), None
        self.federate, self.remember_reads = federate, remember_reads
        self._legs, self._cthread = None, None
        if warm: self.open_vault()

    # -- the vault itself ----------------------------------------------------
    def open_vault(self, wait=False):
        "Open the vault in a daemon thread, once. `wait=True` for a caller that needs it now."
        if self._vthread is None or not self._vthread.is_alive():
            self._vthread = threading.Thread(target=lambda: self._open(), name='ramabana-vault', daemon=True)
            self._vthread.start()
        if wait: self._vthread.join()
        return self

    def _open(self):
        with self._vlock:
            v = self._vault
            if v is None or isinstance(v, (str, Path)):
                from vishalakshi import Vault
                self._vault = v = Vault(str(v)) if v is not None else Vault()
            return v

    @property
    def vault(self):
        "The `Vault`, opened on first use."
        return self._open()

    def connect(self, wait=False):
        """Rebuild the entity graph `related` walks, in a daemon thread, once at a time.

        `Vault.connect` reads the whole store. `poll`'s docstring already says that is right
        for a nightly cron and wrong for a tool call somebody is waiting on -- and then
        `research` ran it inline anyway, on the one path where the user is definitely
        waiting, having just sat through five pages being fetched and filed.

        So it is a background job wherever it is called from. What `related` walks is then a
        graph that is at most one query stale, which is the cost of the thing; a turn held
        open for a full rebuild is not.
        """
        if self._cthread is None or not self._cthread.is_alive():
            def run():
                try: self.vault.connect()
                except Exception as e: self.note(f'could not rebuild the memory graph: {agent_err(e)}')
            self._cthread = threading.Thread(target=run, name='ramabana-vault-connect', daemon=True)
            self._cthread.start()
        if wait: self._cthread.join()
        return self

    # -- durable research memory --------------------------------------------
    def memory_search(self, query, limit=MEM_SECTIONS):
        """Whole sections, plus what they connect to.

        `related` is the part worth having: sections reached along the entity graph or by
        embedding similarity rather than by matching the query. It is how a search finds the
        note you forgot you wrote.
        """
        c = self.vault.context(str(query), sections=int(limit), related=max(2, int(limit) // 2))
        return dict(query=c.query, encoder=c.encoder,
                    results=[_sect(s) for s in c.results],
                    related=[dict(_sect(s), via=s.get('via')) for s in c.related])

    def memory_tree(self, document=''):
        "The heading tree of everything remembered; `document` narrows to one title or doc id."
        toc, d = self.vault.toc(), str(document).strip().lower()
        if d: toc = [t for t in toc if d in str(t.get('title', '')).lower() or d == t.get('doc_id')]
        return [dict(doc_id=t.get('doc_id'), title=t.get('title'), source=t.get('source'),
                     pages=t.get('pages'), tree=_trim(t.get('tree'))) for t in toc]

    def memory_read(self, node_id): return self.vault.read(str(node_id))

    def memory_topics(self, limit=12):
        m = self.vault.map()
        return dict(method=m.get('method'), note=m.get('note'),
                    clusters=list(m.get('clusters') or [])[:int(limit)])

    def memory_forget(self, doc_id):
        self.vault.forget(str(doc_id))
        return True

    # -- seeing the code -----------------------------------------------------
    def search(self, query, limit=20):
        """The code index, the files on disk, and everything ever read -- one ranking.

        Reciprocal rank fusion, not a merged distance: the vault embeds prose, kosha embeds
        identifiers and ripgrep embeds nothing, so an ordering is the only thing the three legs
        have in common. Each leg is tried independently and a leg that fails is reported rather
        than raised, which is what makes this safe to put in front of `search_code` -- a vault
        that will not open degrades to exactly what `LocalHost` already did.
        """
        if not self.federate: return super().search(query, limit)
        try:
            r = self.vault.federate(str(query), limit=int(limit), prose=True, repo=True,
                                    grep=True, dir=self.roots[0])
        except Exception as e:
            self._legs = {'federate_error': agent_err(e)}
            return super().search(query, limit)
        self._legs = dict(r.legs)
        return [_fed_hit(h) for h in r.hits] or super().search(query, limit)

    @property
    def search_note(self):
        if not self.federate: return super().search_note
        if not self._legs: return 'federated: vault prose, kosha index, ripgrep'
        return 'federated: ' + ', '.join(f'{k}={v}' for k, v in self._legs.items())

    # -- reading the web -----------------------------------------------------
    def read_url(self, url, remember=True):
        """Read a page, and keep it unless asked not to.

        `remember=False` is honoured literally -- the page is fetched and returned and never
        touches the vault, which is what it is for. Filing failing is not a read failing, so
        the text is returned either way.
        """
        d = super().read_url(url, remember=remember)
        if d is not None and remember and self.remember_reads:
            try:
                from vishalakshi.acquire import md_title
                self.vault.add(str(d.text), md_title(str(d.text)) or str(url), source=str(url),
                               kind='web', meta=dict(url=str(url), fetched_at=time.time()))
            except Exception as e: self.note(f'could not file {url}: {agent_err(e)}')
        return d

    def research(self, query):
        """Search, read every source into the vault, then answer out of the vault.

        One network pass rather than two: `Vault.web` *is* fossick's `research` with the sources
        kept, so the digest is assembled by asking the vault the same question immediately
        afterwards. What the model reads and what a later session can recall are then the same
        text, instead of a summary of pages nobody kept.
        """
        v, q = self.vault, str(query)
        r = v.web(q, n=5)
        self.connect()            # the graph is what `related` walks; rebuild after the batch, off the turn
        c = v.context(q, sections=MEM_SECTIONS, related=4)
        head = f'searched the web for {q!r}; filed {len(r.added)} of {r.n_found} sources in the vault'
        return '\n\n'.join([head] + [f"## {s['breadcrumb']}\n\n{s['text']}" for s in c.results])

    @property
    def research_note(self): return f'fossick, filed in {Path(self.vault.path).name}'

    # -- standing interests --------------------------------------------------
    def remember(self, text, title=None, tags=()):
        return self.vault.note(str(text), title=title, tags=list(tags or []))

    def watch(self, target, action='remind', every='1d', note=None, **params):
        return self.vault.watch(str(target), action=action, every=every, note=note, **params)

    def watches(self, due_only=False):
        return [dict(w) for w in self.vault.watches(due_only=bool(due_only))]

    def unwatch(self, watch_id): return self.vault.unwatch(str(watch_id))

    def poll(self):
        """Fire everything due. `connect=False`: the graph rebuild is the expensive part.

        `Vault.poll` rebuilds the entity graph when anything succeeded, which reads the whole
        store -- right for a nightly cron, wrong for a tool call the user is waiting on. So
        the tick returns as soon as the watches have run, and the rebuild it earned goes to
        `connect`, which is a background thread and coalesces with `research`'s.
        """
        r = self.vault.poll(connect=False)
        if r.get('ran'): self.connect()
        return r

    @property
    def watch_actions(self):
        from vishalakshi.acquire import ACTIONS
        return ACTIONS

In [ ]:
#| hide
show_doc(VaultHost)

A vault of its own, in a temp dir, so nothing here touches the real one.

In [ ]:
tmp = Path(tempfile.mkdtemp())
host = VaultHost(roots=[tmp], vault=tmp/'vault.db', index=False, web=False)
host.open_vault(wait=True)
host.vault.stats()['encoder']

## Memory

`remember` writes a note; the five memory tools read it back. Notes are ordinary documents
in the same store as everything else, which is why a conclusion comes back next to the evidence
it was drawn from.

In [ ]:
host.remember('The weekly Coles order is milk, sourdough, eggs, baby spinach and olive oil.',
              title='weekly Coles order', tags=['groceries'])
host.remember('Ceres Fair Food delivers a mixed seasonal fruit box on Thursdays.',
              title='Ceres fruit box', tags=['groceries'])

r = host.memory_search('what do I need from the supermarket', limit=2)
[s['title'] for s in r['results']]

In [ ]:
test_eq(len(host.memory_tree()), 2)
test_eq(len(host.memory_tree('ceres')), 1)

node = r['results'][0]['node_id']
'olive oil' in host.memory_read(node)['text'] or 'fruit box' in host.memory_read(node)['text']

And the group is now offered, which is the whole point of putting a vault behind the host:
`tools_for` probes `memory_tree` and drops all five tools when it raises.

In [ ]:
from ramabana.tools import tools_for, NullHost
have = lambda h: {getattr(t, '__name__', '') for t in tools_for(h)}
test_eq('memory_search' in have(NullHost()), False)
test_eq('memory_search' in have(host), True)

## Federated search

`search` asks three indexes one question. Here only two legs can answer -- there is no
kosha index over an empty temp dir -- and `legs` says so rather than pretending otherwise.

In [ ]:
hits = host.search('seasonal fruit box', limit=5)
host.search_note

In [ ]:
#| hide
test_eq(any('fruit box' in h.text.lower() or 'fruit box' in h.path.lower() for h in hits), True)

A prose hit's `symbol` is its `node_id`, so a federated result is directly readable:
`memory_read(hit.symbol)`. A repo hit's is its `mod_name`, for `symbols`. Same field, whichever
follow-up the leg implies.

In [ ]:
prose = [h for h in hits if h.symbol.count('#') == 1]
prose and host.memory_read(prose[0].symbol)['text'][:60]

`federate=False` turns the whole thing off and leaves `LocalHost`'s own search in place --
worth having, because a vault that will not open should degrade rather than fail.

In [ ]:
plain = VaultHost(roots=[tmp], vault=tmp/'vault.db', index=False, web=False,
                  federate=False, warm=False)
test_eq(plain.search_note, plain.__class__.__mro__[1].search_note.fget(plain))

## Watches and reminders

A watch is a job with an interval. `action='remind'` is the degenerate one: when it comes
due it files its own text back into the vault as a note, so the reminder is searchable next to
everything else rather than living in a notification queue the harness does not own.

In [ ]:
w = host.watch('Add the weekly groceries to the Coles trolley, and a Ceres fruit box.',
               action='remind', every='1w', note='weekly shop')
[(x['id'], x['action'], int(x['every'])) for x in host.watches()]

Nothing is due yet -- the first run is scheduled for now, so `poll` at a moment strictly
before that finds nothing.

In [ ]:
test_eq(host.vault.poll(at=w['next_run'] - 1, connect=False)['ran'], 0)

When it does come due, `poll` fires it, and what fired is now findable.

In [ ]:
fired = host.vault.poll(at=w['next_run'] + 1, connect=False)
test_eq(fired['ran'], 1)
test_eq(fired['results'][0]['status'], 'ok')

hit = host.memory_search('what am I supposed to be doing about groceries', limit=3)
[s['title'] for s in hit['results']]

One dead watch must not silence the others: `run_watch` records the failure on the row and
returns it rather than raising, so a poll that fires five watches services all five. Here
the `url` leg is replaced with one that always fails, sitting next to a reminder that works
-- `run_watch` dispatches on the action name, so assigning the attribute is enough.

In [ ]:
def _down(target, **kw): raise ConnectionError('name not resolved')
host.vault.url = _down

dead = host.watch('https://example.invalid/x', action='url', every='1d')
live = host.watch('check which day the Ceres box arrives', action='remind', every='1d')
out = host.vault.poll(at=max(dead['next_run'], live['next_run']) + 1, connect=False)
{r['action']: r['status'] for r in out['results']}

In [ ]:
#| hide
test_eq({r['action']: r['status'] for r in out['results']}, {'url': 'error', 'remind': 'ok'})
test_eq(next(w for w in host.watches() if w['id'] == dead['id'])['last_status'], 'error')
del host.vault.url

The failed watch is still there, rescheduled, with its last status on it -- a watch that
failed once is not a watch you meant to delete.

In [ ]:
host.unwatch(dead['id'])
host.unwatch(live['id'])
test_eq(len(host.watches()), 1)

And the tools follow the capability, the same way the memory ones do.

In [ ]:
tools = have(host)
test_eq({'remember', 'set_reminder', 'watch_url', 'list_watches', 'cancel_watch', 'poll_watches'} <= tools, True)
test_eq(any(t.startswith('poll_') for t in have(NullHost())), False)

`poll_watches` is what the agent actually calls. It reports rather than raises, because a
turn that begins by asking "what is outstanding" should not end there.

In [ ]:
from ramabana.tools import watch_tools
poll_watches = next(t for t in watch_tools(host) if t.__name__ == 'poll_watches')
poll_watches()

## Using it

```python
from ramabana import Agent
from ramabana.vault import VaultHost

host = VaultHost(roots=['~/code/myproject'])     # the shared ~/.vishalakshi/vault.db
agent = Agent(host, model='gpt-mini')
```

That agent has twenty-odd tools instead of the usual dozen: the memory five, the watch six, and
a `search_code` that answers from the repo *and* from everything the vault has ever read. Nothing
about `Agent` changed -- the capabilities arrived because the host has them, which is the only
extension mechanism `Host` has ever needed.